In [ ]:

import warnings
from pathlib import Path

import xarray as xr
from matplotlib import pyplot as plt

from imagematerials.buildings.constants import SCENARIO_SELECT
from imagematerials.buildings.preprocessing.floorspace import (
    compute_average_m2_capita,
    compute_housing_residential,
    compute_housing_type,
    extrapolate_floorspace,
    get_image_floorspace,
)
from imagematerials.buildings.preprocessing.lifetimes import compute_lifetimes
from imagematerials.buildings.preprocessing.population import compute_population, compute_total_population, compute_rur_urb_pop
from imagematerials.util import merge_dims, dataset_to_array


In [ ]:
import warnings
from pathlib import Path

import xarray as xr
import numpy as np

from imagematerials.constants import IMAGE_REGIONS

from imagematerials.buildings.constants import SCENARIO_SELECT
from imagematerials.buildings.preprocessing.floorspace import (
    compute_average_m2_capita,
    compute_housing_residential,
    compute_housing_type,
    extrapolate_floorspace,
    get_image_floorspace
)
from imagematerials.buildings.preprocessing.lifetimes import compute_lifetimes
from imagematerials.buildings.preprocessing.materials import (
    compute_mat_intensities_commercial,
    compute_mat_intensities_residential,
)
from imagematerials.buildings.preprocessing.population import compute_population
from imagematerials.concepts import create_building_graph
from imagematerials.concepts import create_region_graph

from imagematerials.buildings.preprocessing.circular_economy_measures import apply_circular_economy_commercial_floorspace


In [ ]:
base_directory = Path("..", "data", "raw")

database_directory = base_directory / "buildings" / SCENARIO_SELECT
image_directory = base_directory / "image" / SCENARIO_SELECT

In [ ]:
# Get floorspace for commercial + urban/rural
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    floorspace_image_commercial_rururb, minimum_comm = get_image_floorspace(image_directory, base_directory)
floorspace_commercial_rururb = extrapolate_floorspace(floorspace_image_commercial_rururb, minimum_comm)


In [ ]:


# Rural/Urban floorspace [Time, Region, Area]
floorspace_rururb = floorspace_commercial_rururb.sel({"Type": ["Urban", "Rural"]}).rename({"Type": "Area"})

# Commercial floorspace [Time, Region, Type]
floorspace_commercial = floorspace_commercial_rururb.sel(
    {"Type": [x.values for x in floorspace_commercial_rururb.coords["Type"] if x.values not in ["Urban", "Rural"]]})



In [ ]:
print(floorspace_commercial.pint.units)

In [ ]:
# Rural/Urban floorspace [Year, Region, Area]

floorspace_rururb.sel(Area="Urban", Region = '11').plot(label="Urban")
floorspace_rururb.sel(Area="Rural", Region = '11').plot(label="Rural")
floorspace_rururb.sel(Region = '11').sum(["Area"]).plot(label="total residential")
floorspace_commercial.sel(Region = '11').sum(["Type"]).plot(label="total Commercial")
plt.title("Floorspace Urban/Rural")
plt.legend()
plt.show()

In [ ]:
# Calculate population ("Total", "Rural", "Urban")
urban_pop_total, rural_pop_total = compute_rur_urb_pop(image_directory, base_directory)
population = compute_population(image_directory, base_directory)

In [ ]:
# Get the population data [Year, Region, Area].
population = compute_population(image_directory, base_directory)
population.sum("Region").sel(Area="Urban").plot(label="Urban")
population.sum("Region").sel(Area="Rural").plot(label="Rural")
population.sum("Region").sel(Area="Total").plot(label="Total")
plt.title("Population (in millions)")
plt.legend()
plt.show()

In [ ]:
# Average square meter per capita split by residential type [Region, Area, Type]
average_m2_capita = compute_average_m2_capita(base_directory)
print('average_m2_capita:', average_m2_capita.pint.units)
housing_type = compute_housing_type(database_directory)

In [ ]:
# Floorspace m2 for residential buildings [Year, Region, Area, Type]
floorspace_residential = compute_housing_residential(population, 
                                                     average_m2_capita, 
                                                     housing_type, 
                                                     floorspace_rururb, {"test":None})

In [ ]:
# Residential housing type shares [Year, Region, Area, Type]
housing_type = compute_housing_type(database_directory)
for res_type in housing_type.coords["Type"].values:
    housing_type.mean(["Region", "Area"]).sel(Type=res_type).plot(label=res_type)
plt.title("Residential housing shares per type.")
plt.legend()
plt.show()

In [ ]:


for res_type in floorspace_residential.coords["Type"].values:
    floorspace_residential.sum(["Region"]).sel(Type=res_type).plot(label=res_type)
plt.title("Residential housing million m2 per type.")
plt.legend()
plt.show()

In [ ]:
floorspace_commercial_total = floorspace_commercial * population.sel({"Area": "Total"})


In [ ]:
circular_economy_config = {"test": None}

floorspace_commercial_total = floorspace_commercial_total.drop_vars("Area")
floorspace = xr.concat((floorspace_residential, floorspace_commercial_total), dim="Type")

# Lifetime computations, see lifetimes.py

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    lifetimes = compute_lifetimes(base_directory, floorspace_commercial.coords["Type"].values, circular_economy_config)

mat_intensities_comm = compute_mat_intensities_commercial(database_directory, circular_economy_config)
mat_intensities_res = compute_mat_intensities_residential(database_directory, circular_economy_config)
mat_intensities = xr.concat((mat_intensities_res, mat_intensities_comm), dim="Type")
knowledge_graph = create_building_graph()
mat_intensities = knowledge_graph.rebroadcast_xarray(
                        mat_intensities, floorspace.coords["Type"].values)

#TODO remove this quick fix
region_coords = np.sort(floorspace.coords["Region"].values.astype(int)).astype(str)



In [ ]:
import prism
china_p100_mi = prism.Q_(2685, 'kg / m**2')


mat_intensities_comm.loc[dict(Region="20", material="concrete")] = china_p100_mi
# set all values to 2685 only for china
mat_intensities_comm


In [ ]:
lifetimes.get("weibull")
knowledge_graph_region = create_region_graph()


In [ ]:
lifetimes.get("weibull").coords


In [ ]:
lifetime_new = lifetimes.copy()

for key, value in lifetime_new.items():
    lifetime_new[key] = knowledge_graph_region.rebroadcast_xarray(value, output_coords=IMAGE_REGIONS, dim="Region")

In [ ]:
# Returns a single bool
same = lifetime_new["weibull"].identical(lifetimes.get("weibull"))
print('identical:', same)

In [ ]:
import numpy as np

def coords_equal(da1, da2):
    coords1 = da1.coords
    coords2 = da2.coords
    # same coordinate names?
    if set(coords1.keys()) != set(coords2.keys()):
        return False
    # compare each coordinate array
    for k in coords1.keys():
        if not np.array_equal(coords1[k].values, coords2[k].values):
            return False
    return True

print(coords_equal(lifetime_new["weibull"], lifetimes.get("weibull")))

In [ ]:
floorspace